# IMDB Auto-Fill — Model Testing & Evaluation
**Pipeline: EasyOCR (text extraction) + EfficientNet CNN (category classification)**

| Component | Model | Task |
|-----------|-------|------|
| Text Extraction | EasyOCR (CRAFT + CRNN) | Brand, weight, barcode, manufacturer, promotional text |
| Image Classification | EfficientNet-B0 (ImageNet) | Category type & segment type |
| Output | Structured IMDB record | 10 attributes per product |

## 0. Install Dependencies

In [ ]:
# Run this cell once to install required packages
import subprocess, sys
pkgs = ['easyocr', 'torch', 'torchvision', 'opencv-python-headless', 'matplotlib', 'pandas', 'openpyxl', 'Pillow']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)
print('All packages ready')

## 1. Imports & Setup

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from datetime import datetime
warnings.filterwarnings('ignore')

import torch
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import easyocr

IMAGE_DIR = 'product_images'
DEMO_IMAGES = [
    'S221234199_550719011.jpg',
    'S221712802_552034736.jpg',
    'S222711495_554639782.jpg',
    'S222985766_556022646.jpg',
]

IMDB_COLS = ['barcode','category_type','segment_type','manufacturer',
             'brand','product_name','weight_and_unit','packaging_type',
             'country_of_origin','promotional_messages']

LABELS = {
    'barcode':'Barcode','category_type':'Category Type','segment_type':'Segment Type',
    'manufacturer':'Manufacturer','brand':'Brand','product_name':'Product Name',
    'weight_and_unit':'Weight & Unit','packaging_type':'Packaging Type',
    'country_of_origin':'Country of Origin','promotional_messages':'Promotional Messages'
}
print('Setup complete | GPU available:', torch.cuda.is_available())

## 2. Dataset Overview

In [ ]:
all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')]
print(f'Total images in dataset : {len(all_images)}')
print(f'Demo images selected    : {len(DEMO_IMAGES)}')

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Product Images — Ghanaian FMCG Dataset', fontsize=14, fontweight='bold')
titles = ['MOK Fine Soap Rose', "Mummy's Kitchen Seasoning", 'U-Fresh Orange Juice', 'This Way Chocolate Drink']
for ax, fname, title in zip(axes, DEMO_IMAGES, titles):
    img = mpimg.imread(os.path.join(IMAGE_DIR, fname))
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_products.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. EasyOCR — Text Extraction
**Model architecture**: CRAFT (text detection) + CRNN (text recognition)  
Extracts all visible text from the product label.

In [ ]:
print('Loading EasyOCR model (CRAFT + CRNN)...')
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(), verbose=False)
print('EasyOCR ready')

In [ ]:
def extract_text_ocr(image_path):
    """Run EasyOCR on an image and return list of (text, confidence) tuples."""
    results = ocr_reader.readtext(image_path, detail=1, paragraph=False)
    return [(r[1].strip(), round(r[2] * 100, 1)) for r in results if r[2] > 0.3]

def parse_imdb_from_text(texts):
    """Parse OCR text into IMDB fields using pattern matching."""
    all_text = ' '.join([t for t, _ in texts])
    lines = [t for t, _ in texts]
    avg_conf = round(sum(c for _, c in texts) / len(texts), 1) if texts else 0

    # Barcode: 8-13 digit number
    barcode_match = re.search(r'\b(\d{8,13})\b', all_text)
    barcode = barcode_match.group(1) if barcode_match else 'N/A'

    # Weight: number followed by g/ml/kg/l
    weight_match = re.search(r'(\d+\.?\d*)\s*(g|ml|kg|l|G|ML|KG|L)\b', all_text, re.IGNORECASE)
    weight = (weight_match.group(1) + weight_match.group(2).lower()) if weight_match else 'N/A'

    # Manufacturer: line after 'MANUFACTURED BY' or 'MFD BY'
    mfr_match = re.search(r'(?:manufactured by|mfd by|mfr)[:\s]+([A-Za-z][\w\s]+(?:limited|ltd|co|company|inc)?)', all_text, re.IGNORECASE)
    manufacturer = mfr_match.group(1).strip().title() if mfr_match else 'N/A'

    # Country: 'Made in X' or 'Country of origin: X'
    country_match = re.search(r'(?:made in|country of origin)[:\s]+([A-Za-z\s]+)', all_text, re.IGNORECASE)
    country = country_match.group(1).strip().title() if country_match else 'N/A'

    # Packaging type keywords
    pkg_map = {'bottle': 'Bottle', 'sachet': 'Sachet', 'can': 'Can',
               'box': 'Cardboard Box', 'carton': 'Carton', 'pouch': 'Pouch',
               'jar': 'Jar', 'tube': 'Tube', 'bag': 'Bag'}
    packaging = 'N/A'
    for kw, label in pkg_map.items():
        if kw in all_text.lower():
            packaging = label
            break

    # Brand: typically the largest/first text
    brand = lines[0].title() if lines else 'N/A'

    # Product name: first 2 lines joined
    product_name = ' '.join(lines[:2]).title() if len(lines) >= 2 else (lines[0].title() if lines else 'N/A')

    # Promotional messages: non-numeric, longer lines that don't match other fields
    promo_lines = [t for t, c in texts
                   if len(t) > 8 and not re.match(r'^[\d\s]+$', t)
                   and t not in [brand, product_name]]
    promo = promo_lines[0] if promo_lines else 'N/A'

    return {
        'barcode': barcode, 'weight_and_unit': weight,
        'manufacturer': manufacturer, 'country_of_origin': country,
        'packaging_type': packaging, 'brand': brand,
        'product_name': product_name, 'promotional_messages': promo,
        '_ocr_confidence': avg_conf, '_ocr_text': lines
    }

print('OCR parsing functions defined')

In [ ]:
print('Running EasyOCR on product images...')
ocr_results = []
for fname in DEMO_IMAGES:
    path = os.path.join(IMAGE_DIR, fname)
    texts = extract_text_ocr(path)
    parsed = parse_imdb_from_text(texts)
    parsed['image'] = fname
    ocr_results.append(parsed)
    print(f'  {fname}: {len(texts)} text regions detected, avg conf {parsed["_ocr_confidence"]}%')

print('OCR extraction complete')

In [ ]:
# Visualise OCR detections on one image
import cv2
sample_path = os.path.join(IMAGE_DIR, DEMO_IMAGES[0])
ocr_raw = ocr_reader.readtext(sample_path, detail=1)

img_cv = cv2.imread(sample_path)
img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
for (bbox, text, conf) in ocr_raw:
    if conf > 0.3:
        pts = np.array(bbox, np.int32).reshape((-1, 1, 2))
        cv2.polylines(img_cv, [pts], True, (0, 200, 0), 2)
        cv2.putText(img_cv, f'{text} ({conf:.0%})',
                    tuple(map(int, bbox[0])), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

plt.figure(figsize=(10, 7))
plt.imshow(img_cv)
plt.title('EasyOCR Text Detection — MOK Fine Soap Rose', fontweight='bold')
plt.axis('off')
plt.savefig('ocr_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. EfficientNet-B0 CNN — Category & Segment Classification
**Model**: EfficientNet-B0 pre-trained on ImageNet-1K (5.3M parameters)  
**Task**: Classify product category type and segment type from the image

In [ ]:
print('Loading EfficientNet-B0 (ImageNet pre-trained)...')
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
cnn_model = efficientnet_b0(weights=weights)
cnn_model.eval()

imagenet_classes = weights.meta['categories']

# Map ImageNet top predictions → FMCG category & segment
CATEGORY_MAP = {
    # Personal Care
    'soap': ('Personal Care', 'Bar Soap'),
    'lotion': ('Personal Care', 'Body Lotion'),
    'shampoo': ('Personal Care', 'Hair Care'),
    'toothbrush': ('Personal Care', 'Oral Care'),
    'perfume': ('Personal Care', 'Fragrance'),
    'sunscreen': ('Personal Care', 'Skin Care'),
    'lipstick': ('Personal Care', 'Cosmetics'),
    # Food & Beverage
    'bottle': ('Food & Beverage', 'Juice Drinks'),
    'orange': ('Food & Beverage', 'Juice Drinks'),
    'lemon': ('Food & Beverage', 'Juice Drinks'),
    'coffee': ('Food & Beverage', 'Hot Beverages'),
    'cup': ('Food & Beverage', 'Hot Beverages'),
    'chocolate': ('Food & Beverage', 'Chocolate Drinks'),
    'candy': ('Food & Beverage', 'Confectionery'),
    'bread': ('Food & Beverage', 'Bakery'),
    'spice': ('Food & Beverage', 'Seasonings & Spices'),
    'salt': ('Food & Beverage', 'Seasonings & Spices'),
    'sauce': ('Food & Beverage', 'Sauces & Condiments'),
    'beer': ('Food & Beverage', 'Alcoholic Beverages'),
    'wine': ('Food & Beverage', 'Alcoholic Beverages'),
    'water': ('Food & Beverage', 'Water'),
    'milk': ('Food & Beverage', 'Dairy'),
    'ice cream': ('Food & Beverage', 'Frozen Desserts'),
    'pretzel': ('Food & Beverage', 'Snacks'),
    'packet': ('Food & Beverage', 'Packaged Foods'),
}

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f'EfficientNet-B0 loaded | Parameters: {sum(p.numel() for p in cnn_model.parameters()):,}')

In [ ]:
def classify_product(image_path, top_k=5):
    """Run EfficientNet-B0 and map top-k predictions to FMCG categories."""
    img = Image.open(image_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        logits = cnn_model(tensor)
        probs = torch.softmax(logits, dim=1)[0]

    top_probs, top_idxs = probs.topk(top_k)
    top_preds = [(imagenet_classes[i], round(p.item() * 100, 2))
                 for i, p in zip(top_idxs, top_probs)]

    # Match to FMCG category
    category, segment = 'Food & Beverage', 'General'
    cat_conf = 0
    for cls_name, prob in top_preds:
        cls_lower = cls_name.lower()
        for kw, (cat, seg) in CATEGORY_MAP.items():
            if kw in cls_lower:
                category, segment = cat, seg
                cat_conf = prob
                break
        if cat_conf > 0:
            break

    return {
        'category_type': category,
        'segment_type': segment,
        '_cnn_conf': cat_conf,
        '_top_preds': top_preds
    }

print('Running EfficientNet-B0 on product images...')
cnn_results = []
for fname in DEMO_IMAGES:
    path = os.path.join(IMAGE_DIR, fname)
    res = classify_product(path)
    res['image'] = fname
    cnn_results.append(res)
    print(f'  {fname}: {res["category_type"]} → {res["segment_type"]} (conf {res["_cnn_conf"]}%)')
    print(f'    Top ImageNet predictions: {res["_top_preds"][:3]}')

print('CNN classification complete')

## 5. Merge OCR + CNN → Final IMDB Records

In [ ]:
# Merge OCR text fields + CNN category fields
final_records = []
for ocr, cnn in zip(ocr_results, cnn_results):
    record = {
        'image': ocr['image'],
        'barcode': ocr['barcode'],
        'category_type': cnn['category_type'],
        'segment_type': cnn['segment_type'],
        'manufacturer': ocr['manufacturer'],
        'brand': ocr['brand'],
        'product_name': ocr['product_name'],
        'weight_and_unit': ocr['weight_and_unit'],
        'packaging_type': ocr['packaging_type'],
        'country_of_origin': ocr['country_of_origin'],
        'promotional_messages': ocr['promotional_messages'],
        'ocr_confidence': ocr['_ocr_confidence'],
        'cnn_confidence': cnn['_cnn_conf'],
        'avg_confidence': round((ocr['_ocr_confidence'] + cnn['_cnn_conf']) / 2, 1),
    }
    final_records.append(record)

df_final = pd.DataFrame(final_records)
display_df = df_final[['product_name'] + IMDB_COLS].copy()
display_df.columns = ['Product Name'] + [LABELS[c] for c in IMDB_COLS]
display_df

## 6. Confidence Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Performance Analysis', fontsize=14, fontweight='bold')

# OCR vs CNN confidence per product
ax1 = axes[0]
products = [r['product_name'][:20] + '...' if len(r['product_name']) > 20
            else r['product_name'] for r in final_records]
x = np.arange(len(products))
w = 0.35
ax1.bar(x - w/2, df_final['ocr_confidence'], w, label='EasyOCR', color='#3498db', alpha=0.85)
ax1.bar(x + w/2, df_final['cnn_confidence'], w, label='EfficientNet CNN', color='#e67e22', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(products, rotation=20, ha='right', fontsize=8)
ax1.set_ylabel('Confidence (%)')
ax1.set_title('EasyOCR vs CNN Confidence per Product')
ax1.legend()
ax1.set_ylim(0, 110)
ax1.axhline(75, color='gray', linestyle='--', alpha=0.5, label='75% threshold')

# Overall avg confidence
ax2 = axes[1]
colors = ['#2ecc71' if v >= 75 else '#e74c3c' for v in df_final['avg_confidence']]
bars = ax2.barh(products, df_final['avg_confidence'], color=colors, alpha=0.85)
ax2.set_xlabel('Average Confidence (%)')
ax2.set_title('Overall Model Confidence per Product')
ax2.set_xlim(0, 110)
ax2.axvline(75, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars, df_final['avg_confidence']):
    ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
             f'{val}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Overall EasyOCR avg confidence : {df_final["ocr_confidence"].mean():.1f}%')
print(f'Overall CNN avg confidence     : {df_final["cnn_confidence"].mean():.1f}%')
print(f'Combined avg confidence        : {df_final["avg_confidence"].mean():.1f}%')

## 7. Export Results

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# CSV
csv_path = f'imdb_results_{timestamp}.csv'
df_final[['product_name'] + IMDB_COLS + ['ocr_confidence','cnn_confidence','avg_confidence']].to_csv(csv_path, index=False)
print(f'CSV saved: {csv_path}')

# Excel with two sheets
xlsx_path = f'imdb_results_{timestamp}.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    display_df.to_excel(writer, sheet_name='IMDB Results', index=False)
    df_final[['product_name','ocr_confidence','cnn_confidence','avg_confidence']].to_excel(
        writer, sheet_name='Model Confidence', index=False)
print(f'Excel saved: {xlsx_path}')

print(f'\nFinal Summary')
print(f'  Products processed : {len(df_final)}')
print(f'  Fields per product : {len(IMDB_COLS)}')
print(f'  Avg OCR confidence : {df_final["ocr_confidence"].mean():.1f}%')
print(f'  Avg CNN confidence : {df_final["cnn_confidence"].mean():.1f}%')
print(f'  Avg overall        : {df_final["avg_confidence"].mean():.1f}%')